# 01. Exploratory Data Analysis & Split Validation

This notebook analyzes the **face_attendance_dataset** (CC0, synthetic) and verifies the integrity of the data pipeline and leakage-free splits.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
from fras.training.dataset import validate_dataset, generate_splits

root_dir = Path("..")
raw_dir = root_dir / "data" / "raw" / "face_attendance_dataset"
splits_dir = root_dir / "data" / "splits"

## 1. Dataset Integrity & Schema Validation

In [2]:
stats = validate_dataset(raw_dir)
print(f"Total Persons: {stats.total_persons}")
print(f"Total Images: {stats.total_images}")
print(f"Image Shape: {stats.image_shape}")
print(f"Corrupt Images: {stats.corrupted_images}")
print(f"Attendance Records: {stats.attendance_records}")
print(f"Departments: {stats.departments}")
print(f"Attendance Status Counts: {stats.status_counts}")
print(f"Camera Usage: {stats.camera_counts}")
print(f"Mean Luminance: {stats.mean_luminance:.2f} +/- {stats.std_luminance:.2f}")

## 2. Leakage-Free Partitions (Closed-Set & Open-Set)

In [3]:
closed_split, open_split = generate_splits(raw_dir, splits_dir, seed=42)

# Verify Closed-Set
train_paths = {x['relative_path'] for x in closed_split['splits']['train']}
val_paths = {x['relative_path'] for x in closed_split['splits']['val']}
test_paths = {x['relative_path'] for x in closed_split['splits']['test']}
assert len(train_paths & val_paths) == 0
assert len(train_paths & test_paths) == 0
assert len(val_paths & test_paths) == 0
print("Closed-Set: Verified disjoint (300 train / 60 val / 90 test)")

# Verify Open-Set (Unseen identities)
open_train_ids = set(open_split['identities']['train'])
open_val_ids = set(open_split['identities']['val'])
open_test_ids = set(open_split['identities']['test'])
assert len(open_train_ids & open_val_ids) == 0
assert len(open_train_ids & open_test_ids) == 0
assert len(open_val_ids & open_test_ids) == 0
print("Open-Set: Verified strictly disjoint (20 train IDs / 5 val IDs / 5 test IDs)")